# Mezcla sistemática de audios


In [ ]:
"""Restricciones:
  - 400 archivos de speech disponibles
  - 5 niveles de SNR: 0, 5, 10, 15, 20 dB
  - Split por archivo de speech: 70/15/15 (train/val/test)
  - Mismo speech aparece en los 5 niveles → comparación justa entre niveles
  - Contaminante: solo noise
  - Sin repetición consecutiva del mismo contaminante dentro de un nivel
"""
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
import random
import json
from collections import defaultdict
from tqdm import tqdm

In [ ]:
# Configuración
semilla = 42 # Para reproducibilidad
random.seed(semilla)
np.random.seed(semilla)

SR         = 16000
SNR_LEVELS = [0, 5, 10, 15, 20]
SPLIT      = (0.70, 0.15, 0.15)   # train - val - test

In [ ]:
#convierte los .wav en 16khz
def load_audio(path, sr=SR):
    audio, _ = librosa.load(str(path), sr=sr, mono=True)
    return audio.astype(np.float32)

#calcula el rms / como el volumen promedio 
def rms(x):
    return float(np.sqrt(np.mean(x ** 2) + 1e-9))

#mezcla noise y speech
def mix_at_snr(speech, contaminant, snr_db):
    """
    Mezcla speech + contaminant al SNR objetivo.
    
    Si el contaminante es más corto que el speech, se repite.
    Se toma un offset aleatorio para no usar siempre el inicio.
    La mezcla se normaliza al final para evitar clipping.
    
    """
    #verifica la longitud del noise
    if len(contaminant) < len(speech):
        reps = int(np.ceil(len(speech) / len(contaminant)))
        contaminant = np.tile(contaminant, reps)
 
    max_start = len(contaminant) - len(speech)
    start = random.randint(0, max_start) if max_start > 0 else 0
    contaminant = contaminant[start : start + len(speech)]
 
    #aqui se escala alpha para alcanzar el SNR objetivo
    alpha = rms(speech) / (rms(contaminant) * 10 ** (snr_db / 20.0))
    mixed = speech + alpha * contaminant
 
    #normaliza el resultado para que ninguno supere 1
    peak = np.max(np.abs(mixed))
    if peak > 0:
        mixed = mixed / peak
 
    #verifica que el snr sea correcto
    snr_real = 20 * np.log10(rms(speech) / (rms(alpha * contaminant) + 1e-9))
 
    return mixed.astype(np.float32), float(alpha), round(float(snr_real), 2)
 

 
def cycle_sample(file_list, n):

    if len(file_list) == 0:
        raise ValueError("La lista de archivos está vacía.")
    full_cycles = n // len(file_list)
    remainder   = n %  len(file_list)
    pool = file_list * full_cycles
    if remainder > 0:
        pool += random.sample(file_list, remainder)
    random.shuffle(pool)
    return pool
 
#divicion de speech en train,test y val
def split_speech(speech_files, split=(0.70, 0.15, 0.15)):
    files = speech_files.copy()
    random.shuffle(files)
    n = len(files)
    n_train = int(n * split[0])              
    n_val   = int(n * split[1])              
    return (
        files[:n_train],
        files[n_train : n_train + n_val],
        files[n_train + n_val :]
    )

#gnera el dataset de las mezclas 
def build_dataset(speech_dir, noise_dir, out_dir):
    """
    Estructura de salida:
        out_dir/
          train/snr_Xdb/mix_XXXX.wav
          val/snr_Xdb/mix_XXXX.wav
          test/snr_Xdb/mix_XXXX.wav
          metadata.json
          split_info.json
    """
    out_dir = Path(out_dir)

    speech_files = sorted(Path(speech_dir).rglob("*.wav"))
    noise_files  = sorted(Path(noise_dir).rglob("*.wav"))
 
    speech_files = [str(p) for p in speech_files]
    noise_files  = [str(p) for p in noise_files]

    speech_files = speech_files[:100] 
 
    print(f"Speech disponible : {len(speech_files):>4} archivos")
    print(f"Noise disponible  : {len(noise_files):>4} archivos")
    print()
 
    if len(speech_files) == 0:
        raise FileNotFoundError(f"No se encontraron .wav en {speech_dir}")
    if len(noise_files) == 0:
        raise FileNotFoundError(f"No se encontraron .wav en {noise_dir}")
 
    train_sp, val_sp, test_sp = split_speech(speech_files, SPLIT)
    split_info = {
        "seed"        : semilla,
        "total_speech": len(speech_files),
        "train"       : len(train_sp),
        "val"         : len(val_sp),
        "test"        : len(test_sp),
        "snr_levels"  : SNR_LEVELS,
        "contaminant" : "noise_only",
    }
    print(f"Split de speech → train: {len(train_sp)} | val: {len(val_sp)} | test: {len(test_sp)}")
    print()
 
    metadata   = []
    snr_errors = defaultdict(list)  
 
    subsets = [("train", train_sp), ("val", val_sp), ("test", test_sp)]
 
    for subset_name, speech_subset in subsets:
        n = len(speech_subset)

        for snr in SNR_LEVELS:
            out_snr_dir = out_dir / subset_name / f"snr_{snr}db"
            out_snr_dir.mkdir(parents=True, exist_ok=True)
 
            noise_pool = cycle_sample(noise_files, n)
 
            for i, (sp_path, noise_path) in enumerate(
                zip(speech_subset, noise_pool)
            ):
                try:
                    speech      = load_audio(sp_path)
                    contaminant = load_audio(noise_path)
                    mixed, alpha, snr_real = mix_at_snr(speech, contaminant, snr)
 
                    out_path = out_snr_dir / f"mix_{i:04d}.wav"
                    sf.write(str(out_path), mixed, SR)
 
                    metadata.append({
                        "file"           : str(out_path),
                        "subset"         : subset_name,
                        "speech_src"     : sp_path,
                        "contaminant_src": noise_path,
                        "contaminant_type": "noise",
                        "snr_target_db"  : snr,
                        "snr_real_db"    : snr_real,
                        "alpha"          : alpha,
                    })
 
                   
                    error = abs(snr_real - snr)
                    if error > 1.0:
                        snr_errors[snr].append({
                            "file"    : str(out_path),
                            "snr_real": snr_real,
                            "error_db": round(error, 2)
                        })
 
                except Exception as e:
                    print(f"  [ERROR] {sp_path} | {noise_path} → {e}")
 
            n_done = len(speech_subset)
            print(f"  [{subset_name:5s}] SNR {snr:>2} dB — {n_done} mezclas")
 
        print()
 
    meta_path = out_dir / "metadata.json"
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)
 
    split_path = out_dir / "split_info.json"
    with open(split_path, "w") as f:
        json.dump(split_info, f, indent=2)
 
    total    = len(metadata)
    snr_warn = sum(len(v) for v in snr_errors.values())
 
    print("=" * 52)
    print(f"  Total de mezclas generadas : {total}")
    print(f"  Contaminante               : noise (100%)")
    print(f"  Mezclas con error SNR >1dB : {snr_warn}")
    print(f"  Metadata guardado en       : {meta_path}")
    print("=" * 52)
 
    return metadata, split_info

In [ ]:
import os
os.chdir(r"D:\archive\musan")

build_dataset(
    speech_dir = "speech",
    noise_dir  = "noise",
    out_dir    = "mixed_audio"
)